In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import random

def calculate_balance_score(label_row, min_classes=3, max_classes=4):
    """
    计算一行的类别均衡度分数，分数越高越均衡
    """
    unique_vals, counts = torch.unique(label_row, return_counts=True)
    total_pixels = len(label_row)

    # 检查类别数量是否符合要求
    n_unique = len(unique_vals)
    if not (min_classes <= n_unique <= max_classes):
        return float('-inf')  # 不符合条件，返回负无穷

    # 计算各类别占比
    ratios = counts.float() / total_pixels

    # 均衡度评分：使用熵或方差的倒数
    # 熵越大，说明分布越均匀
    entropy = -(ratios * torch.log(ratios + 1e-8)).sum()

    # 或者使用方差的倒数（方差越小越均匀，但这里我们希望各类别都有一定比例）
    # 更好的方法是考虑理想均匀分布的偏离程度
    ideal_ratio = 1.0 / n_unique
    deviation_penalty = torch.abs(ratios - ideal_ratio).mean()

    # 综合评分：熵越大越好，偏离越小越好
    score = entropy - deviation_penalty * 2  # 权重可根据需要调整

    return score.item()

def visualize_class_probabilities_with_best_row(image_idx, dataset, model, device):
    """
    主要的可视化函数 - 找到最均衡的行
    """
    # 获取单个样本
    image_tensor, label_tensor, original_image = dataset[image_idx]
    image_batch = original_image.unsqueeze(0).to(device)

    # 模型预测
    with torch.no_grad():
        output = model(image_batch)  # Shape: [1, 5, H, W]
        probabilities = F.softmax(output, dim=1)  # 转换为概率
        prob_squeezed = probabilities.squeeze(0)  # Shape: [5, H, W]
        label_squeezed = label_tensor  # Shape: [H, W]

    H, W = label_squeezed.shape

    # 计算每行的均衡度分数
    row_scores = []
    for h in range(H):
        row_labels = label_squeezed[h, :]  # 当前行的标签
        score = calculate_balance_score(row_labels)
        row_scores.append((h, score))

    # 过滤掉无效行（分数为负无穷的）
    valid_rows = [(h, score) for h, score in row_scores if score != float('-inf')]

    if not valid_rows:
        print("没有找到符合条件的行")
        return None

    # 找到最均衡的行（分数最高的）
    best_row_idx, best_score = max(valid_rows, key=lambda x: x[1])

    print(f"Selected row {best_row_idx} with balance score: {best_score:.3f}")

    # 获取最佳行的数据
    row_labels = label_squeezed[best_row_idx, :]  # [W]
    row_probs = prob_squeezed[:, best_row_idx, :]  # [5, W] - 各类别的概率

    # 创建可视化图表
    fig, ax = plt.subplots(figsize=(15, 8))

    # 绘制各类别的概率曲线
    x_coords = np.arange(W)
    colors = ['red', 'blue', 'green', 'orange', 'purple']
    class_names = [f'Class {i}' for i in range(5)]

    for class_idx in range(5):
        ax.plot(x_coords, row_probs[class_idx, :].cpu().numpy(),
                label=f'{class_names[class_idx]}',
                color=colors[class_idx], linewidth=2, alpha=0.7)

    # 在背景显示真实类别分布
    unique_labels = torch.unique(row_labels)
    for label_val in unique_labels:
        mask = (row_labels == label_val)
        start_pos = None
        for i in range(len(mask)):
            if mask[i]:
                if start_pos is None:
                    start_pos = i
            else:
                if start_pos is not None:
                    ax.axvspan(start_pos, i, alpha=0.2, color=get_label_color(int(label_val)),
                              label=f'True {int(label_val)}' if int(label_val) not in [int(x.get_label().split()[-1]) for x in ax.get_lines()] else "")
                    start_pos = None
        # 处理连续区域在行末的情况
        if start_pos is not None:
            ax.axvspan(start_pos, len(mask), alpha=0.2, color=get_label_color(int(label_val)))

    ax.set_xlabel('Pixel Position (X-axis)')
    ax.set_ylabel('Probability')
    ax.set_title(f'Class Probabilities along Row {best_row_idx}\nTrue Labels shown as background colors\nBalance Score: {best_score:.3f}')
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # 分析各类别的统计特性
    analyze_class_statistics(row_labels, row_probs)

    return best_row_idx, best_score

def get_label_color(label_val):
    """为不同标签分配颜色"""
    colors = ['red', 'blue', 'green', 'orange', 'purple']
    return colors[label_val % len(colors)]

def analyze_class_statistics(row_labels, row_probs):
    """
    分析各类别的统计特性（均值、方差等）
    """
    print("Class Statistics Analysis:")
    print("-" * 60)

    for class_idx in range(5):
        class_probs = row_probs[class_idx, :]
        mean_prob = class_probs.mean().item()
        var_prob = class_probs.var().item()

        # 计算该类别在真实标签中的出现频率
        class_in_true = (row_labels == class_idx).sum().item()
        total_pixels = len(row_labels)
        freq_ratio = class_in_true / total_pixels

        # 计算该类别概率的最大值（反映峰值特征）
        max_prob = class_probs.max().item()

        print(f"Class {class_idx}:")
        print(f"  True frequency ratio: {freq_ratio:.3f}")
        print(f"  Probability mean: {mean_prob:.3f}")
        print(f"  Probability variance: {var_prob:.6f}")
        print(f"  Probability max: {max_prob:.3f}")
        print()

def aggregate_statistics_for_all_images(selected_results):
    """
    汇总所有图片的统计结果
    """
    print("\n" + "="*80)
    print("AGGREGATED STATISTICS ACROSS ALL SELECTED ROWS")
    print("="*80)

    # 收集所有类别的统计数据
    all_stats = {i: {'freq_ratios': [], 'means': [], 'vars': [], 'maxes': []} for i in range(5)}

    for img_idx, (row_idx, score, stats_list) in enumerate(selected_results):
        print(f"\nImage {img_idx}, Row {row_idx}, Balance Score: {score:.3f}")
        for class_idx, stats in enumerate(stats_list):
            freq_ratio, mean_prob, var_prob, max_prob = stats
            all_stats[class_idx]['freq_ratios'].append(freq_ratio)
            all_stats[class_idx]['means'].append(mean_prob)
            all_stats[class_idx]['vars'].append(var_prob)
            all_stats[class_idx]['maxes'].append(max_prob)

            print(f"  Class {class_idx}: Freq={freq_ratio:.3f}, Mean={mean_prob:.3f}, Var={var_prob:.6f}, Max={max_prob:.3f}")

    print("\nOverall Averages:")
    print("-" * 60)
    for class_idx in range(5):
        avg_freq = np.mean(all_stats[class_idx]['freq_ratios'])
        avg_mean = np.mean(all_stats[class_idx]['means'])
        avg_var = np.mean(all_stats[class_idx]['vars'])
        avg_max = np.mean(all_stats[class_idx]['maxes'])

        print(f"Class {class_idx}: Avg_Freq={avg_freq:.3f}, Avg_Mean={avg_mean:.3f}, Avg_Var={avg_var:.6f}, Avg_Max={avg_max:.3f}")

    # 分析你的猜想：高频类别 vs 低频类别的特征
    print("\nTesting Your Hypothesis:")
    print("-" * 60)
    for class_idx in range(5):
        freqs = np.array(all_stats[class_idx]['freq_ratios'])
        means = np.array(all_stats[class_idx]['means'])
        vars_ = np.array(all_stats[class_idx]['vars'])

        # 计算频率与均值、方差的相关性
        if len(freqs) > 1:
            freq_mean_corr = np.corrcoef(freqs, means)[0, 1]
            freq_var_corr = np.corrcoef(freqs, vars_)[0, 1]

            print(f"Class {class_idx}:")
            print(f"  Frequency vs Mean Correlation: {freq_mean_corr:.3f}")
            print(f"  Frequency vs Variance Correlation: {freq_var_corr:.3f}")

    return all_stats

def run_experiment():
    # 你的原始代码
    try:
        import kagglehub
    except ImportError:
        !pip install kagglehub

    from src.dataset.DroneSegDataSet import MyDataset
    from src.dataset.CheckDataset import check_dataset
    from torchvision import transforms

    # 加载数据集
    ds_path = check_dataset()
    dataset = MyDataset(
        'drone_seg_dataset/classes_dataset/classes_dataset/original_images',
        'drone_seg_dataset/classes_dataset/classes_dataset/label_images_semantic',
        transform=transforms.Compose([
            transforms.ToTensor(),
        ]),
    )

    n_classes = 5
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    from src.model.baseline_model.Unet_model import UNet
    model = UNet(in_channels=3, num_classes=n_classes).to(device)

    param_path = 'resources/para/baseline_unet.pth'
    model.load_state_dict(torch.load(param_path, map_location=device))

    # 随机挑选10张图片
    total_images = len(dataset)
    selected_indices = random.sample(range(total_images), min(10, total_images))

    print(f"Randomly selected image indices: {selected_indices}")

    # 存储选中行的结果
    selected_results = []

    for idx, img_idx in enumerate(selected_indices):
        print(f"\n{'='*50}")
        print(f"Processing image {idx+1}/10 (index: {img_idx})")
        print(f"{'='*50}")

        try:
            result = visualize_class_probabilities_with_best_row(img_idx, dataset, model, device)
            if result is not None:
                best_row_idx, best_score = result

                # 重新计算统计信息用于汇总
                image_tensor, label_tensor, original_image = dataset[img_idx]
                image_batch = original_image.unsqueeze(0).to(device)

                with torch.no_grad():
                    output = model(image_batch)
                    probabilities = F.softmax(output, dim=1)
                    prob_squeezed = probabilities.squeeze(0)
                    label_squeezed = label_tensor

                row_labels = label_squeezed[best_row_idx, :]
                row_probs = prob_squeezed[:, best_row_idx, :]

                # 收集统计数据
                stats_list = []
                for class_idx in range(5):
                    class_probs = row_probs[class_idx, :]
                    mean_prob = class_probs.mean().item()
                    var_prob = class_probs.var().item()

                    class_in_true = (row_labels == class_idx).sum().item()
                    total_pixels = len(row_labels)
                    freq_ratio = class_in_true / total_pixels

                    max_prob = class_probs.max().item()

                    stats_list.append((freq_ratio, mean_prob, var_prob, max_prob))

                selected_results.append((img_idx, best_score, stats_list))

        except Exception as e:
            print(f"Error processing image {img_idx}: {e}")
            import traceback
            traceback.print_exc()
            continue

    # 汇总所有结果
    if selected_results:
        aggregate_statistics_for_all_images(selected_results)
    else:
        print("No valid rows found across all images!")

# 运行实验
run_experiment()